In [1]:
import pandas as pd

# Load the GTFS static files
stop_times = pd.read_csv("D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25/stop_times.txt")
trips = pd.read_csv("D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25/trips.txt")
routes = pd.read_csv("D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25/routes.txt")

# Isolate bus routes (route_type == 3)
bus_routes = routes[routes['route_type'] == 3]
bus_trips = trips[trips['route_id'].isin(bus_routes['route_id'])]

# Merge to link stops with their trip and route
schedule = pd.merge(bus_trips, stop_times, on='trip_id')

# Clean time formatting for trips crossing midnight (e.g., 24:15:00)
schedule['arrival_time_clean'] = pd.to_timedelta(
    schedule['arrival_time'].str.replace(r'^\s*24:', '00:', regex=True)
                            .str.replace(r'^\s*25:', '01:', regex=True)
)

# Filter for the first stop of each trip to calculate departure headways
# Note: Ensure you filter by a specific service_id (e.g., a standard weekday)
first_stops = schedule[schedule['stop_sequence'] == 1].copy()
first_stops = first_stops[first_stops['service_id'] == 5] # Replace '5' with the active weekday service_id

# Sort chronologically and calculate the time difference (headway)
first_stops = first_stops.sort_values(by=['route_id', 'direction_id', 'arrival_time_clean'])
first_stops['headway_mins'] = first_stops.groupby(['route_id', 'direction_id'])['arrival_time_clean'].diff().dt.total_seconds() / 60

# Aggregate to find the median scheduled headway per route
route_baselines = first_stops.groupby('route_id')['headway_mins'].median().reset_index()

# Merge back the route names for readability and export
route_baselines = pd.merge(route_baselines, bus_routes[['route_id', 'route_short_name', 'route_long_name']], on='route_id')
route_baselines.to_csv('scheduled_bus_baselines.csv', index=False)
print("Static baselines successfully generated.")

Static baselines successfully generated.


In [1]:
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
from google.transit import gtfs_realtime_pb2
import os

URL = "https://gtfs-rt.itsmarta.com/TMGTFSRealTimeWebService/vehicle"
LOG_FILE = "marta_vehicle_telemetry.csv"

# Configure a robust session with retries and exponential backoff
session = requests.Session()
retry = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)

def poll_marta_telemetry():
    feed = gtfs_realtime_pb2.FeedMessage()
    
    try:
        # Added a longer timeout to accommodate slow server responses
        response = session.get(URL, timeout=15)
        response.raise_for_status()
        
        feed.ParseFromString(response.content)
        records = []
        
        for entity in feed.entity:
            if entity.HasField('vehicle'):
                v = entity.vehicle
                records.append({
                    "vehicle_id": v.vehicle.id,
                    "trip_id": v.trip.trip_id,
                    "route_id": v.trip.route_id,
                    "latitude": v.position.latitude,
                    "longitude": v.position.longitude,
                    "timestamp": v.timestamp
                })
        
        if records:
            df = pd.DataFrame(records)
            file_exists = os.path.isfile(LOG_FILE)
            df.to_csv(LOG_FILE, mode='a', index=False, header=not file_exists)
            print(f"[{pd.Timestamp.now()}] Success: Logged {len(records)} vehicles.")
        else:
            print(f"[{pd.Timestamp.now()}] API connected, but no vehicles active.")
            
    except requests.exceptions.ConnectionError as ce:
        print(f"[{pd.Timestamp.now()}] Connection Error (Server Down): {ce}")
    except requests.exceptions.Timeout:
        print(f"[{pd.Timestamp.now()}] Timeout: MARTA server took too long to respond.")
    except Exception as e:
        print(f"[{pd.Timestamp.now()}] Unexpected Error: {e}")

if __name__ == "__main__":
    print("Starting MARTA GTFS-RT Polling Service...")
    while True:
        poll_marta_telemetry()
        # Sleep for 60 seconds before pinging again
        time.sleep(60)

Starting MARTA GTFS-RT Polling Service...
[2026-10-02 00:45:38.709922] Connection Error (Server Down): HTTPConnectionPool(host='gtfs-rt.itsmarta.com', port=80): Max retries exceeded with url: /TMGTFSRealTimeWebService/vehicle/ (Caused by ConnectTimeoutError(<HTTPConnection(host='gtfs-rt.itsmarta.com', port=80) at 0x137dfe028b0>, 'Connection to gtfs-rt.itsmarta.com timed out. (connect timeout=15)'))
[2026-10-02 00:47:45.823786] Connection Error (Server Down): HTTPConnectionPool(host='gtfs-rt.itsmarta.com', port=80): Max retries exceeded with url: /TMGTFSRealTimeWebService/vehicle/ (Caused by ConnectTimeoutError(<HTTPConnection(host='gtfs-rt.itsmarta.com', port=80) at 0x137dfe3d7b0>, 'Connection to gtfs-rt.itsmarta.com timed out. (connect timeout=15)'))
[2026-10-02 00:49:52.087295] Connection Error (Server Down): HTTPConnectionPool(host='gtfs-rt.itsmarta.com', port=80): Max retries exceeded with url: /TMGTFSRealTimeWebService/vehicle/ (Caused by ConnectTimeoutError(<HTTPConnection(host='

KeyboardInterrupt: 

In [1]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString

# Set the path to your GTFS directory
GTFS_DIR = r"D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"

# Load the primary tables
routes = pd.read_csv(os.path.join(GTFS_DIR, "routes.txt"))
trips = pd.read_csv(os.path.join(GTFS_DIR, "trips.txt"))
stops = pd.read_csv(os.path.join(GTFS_DIR, "stops.txt"))
calendar = pd.read_csv(os.path.join(GTFS_DIR, "calendar.txt"))

# Filter for bus routes only (route_type == 3)
bus_routes = routes[routes["route_type"] == 3]
print(f"Total Bus Routes: {len(bus_routes)}")
print(f"Total Stops: {len(stops)}")

Total Bus Routes: 113
Total Stops: 8724


In [2]:
import os
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Configuration & Path Setup
# ---------------------------------------------------------
# Update this path if running from a different working directory
GTFS_DIR = r"D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"
OUTPUT_CSV = "weekday_bus_headways.csv"

# ---------------------------------------------------------
# 2. Data Ingestion
# ---------------------------------------------------------
print("Loading GTFS text files...")
calendar = pd.read_csv(os.path.join(GTFS_DIR, "calendar.txt"))
routes = pd.read_csv(os.path.join(GTFS_DIR, "routes.txt"))
trips = pd.read_csv(os.path.join(GTFS_DIR, "trips.txt"))
stop_times = pd.read_csv(os.path.join(GTFS_DIR, "stop_times.txt"))

# ---------------------------------------------------------
# 3. Filter for Active Weekday Bus Operations
# ---------------------------------------------------------
# Identify active weekday service IDs (MARTA standard weekday service_id is typically 5)
weekday_service_ids = calendar[calendar["monday"] == 1]["service_id"].tolist()
print(f"Active weekday service IDs: {weekday_service_ids}")

# Filter for bus routes only (route_type == 3)
bus_routes = routes[routes["route_type"] == 3].copy()
bus_trips = trips[
    (trips["route_id"].isin(bus_routes["route_id"])) & 
    (trips["service_id"].isin(weekday_service_ids))
].copy()

print(f"Loaded {len(bus_routes)} bus routes and {len(bus_trips)} weekday bus trips.")

# ---------------------------------------------------------
# 4. Process Stop Times & Normalize Timestamps
# ---------------------------------------------------------
# Merge trips with stop schedules
schedule = pd.merge(
    bus_trips[["trip_id", "route_id", "direction_id"]],
    stop_times[["trip_id", "arrival_time", "stop_sequence", "stop_id"]],
    on="trip_id"
)

# Convert arrival time strings to Timedelta, handling post-midnight times (> 24:00:00)
def parse_gtfs_time(time_str):
    try:
        parts = [int(p) for p in str(time_str).strip().split(":")]
        h, m, s = parts[0], parts[1], parts[2]
        if h >= 24:
            return pd.to_timedelta(f"{h - 24:02d}:{m:02d}:{s:02d}") + pd.Timedelta(days=1)
        return pd.to_timedelta(f"{h:02d}:{m:02d}:{s:02d}")
    except Exception:
        return pd.NaT

print("Parsing arrival times...")
schedule["arrival_td"] = schedule["arrival_time"].apply(parse_gtfs_time)

# ---------------------------------------------------------
# 5. Extract Trip Departures & Define Time Windows
# ---------------------------------------------------------
# Focus on the initial stop (stop_sequence == 1) for each trip to measure departure headways
first_stops = schedule[schedule["stop_sequence"] == 1].copy()

# Map arrival times into standard transportation planning bins (seconds of day)
seconds_in_day = first_stops["arrival_td"].dt.total_seconds() % 86400

time_bins = [0, 21600, 32400, 54000, 68400, 86400]
time_labels = [
    "Early Morning (00:00-06:00)",
    "AM Peak (06:00-09:00)",
    "Midday (09:00-15:00)",
    "PM Peak (15:00-19:00)",
    "Night (19:00-24:00)"
]

first_stops["time_window"] = pd.cut(
    seconds_in_day,
    bins=time_bins,
    labels=time_labels,
    right=False
)

# ---------------------------------------------------------
# 6. Compute Headways (Frequency in Minutes)
# ---------------------------------------------------------
# Sort chronologically by route, direction, and arrival time
first_stops = first_stops.sort_values(by=["route_id", "direction_id", "arrival_td"])

# Calculate headway as the difference between successive vehicle departures
first_stops["headway_mins"] = (
    first_stops.groupby(["route_id", "direction_id"])["arrival_td"]
    .diff()
    .dt.total_seconds() / 60.0
)

# Filter out erroneous negative values or extreme overnight gaps (> 120 mins)
valid_headways = first_stops[
    (first_stops["headway_mins"] > 0) & 
    (first_stops["headway_mins"] <= 120)
]

# Compute median headway per route and time-of-day window
headway_agg = (
    valid_headways.groupby(["route_id", "time_window"], observed=False)["headway_mins"]
    .median()
    .reset_index()
)

# Pivot to wide format
headways_pivot = headway_agg.pivot(
    index="route_id",
    columns="time_window",
    values="headway_mins"
).reset_index()

# ---------------------------------------------------------
# 7. Merge Route Metadata & Classify Network Tiers
# ---------------------------------------------------------
results = pd.merge(
    bus_routes[["route_id", "route_short_name", "route_long_name"]],
    headways_pivot,
    on="route_id",
    how="inner"
)

# Classify Frequent Transit Network (FTN): Routes maintaining <= 15 min headways in peak periods
results["is_frequent_network"] = (
    (results["AM Peak (06:00-09:00)"] <= 15) & 
    (results["PM Peak (15:00-19:00)"] <= 15)
)

# Sort by route short name
results = results.sort_values(by="route_short_name", key=lambda col: pd.to_numeric(col, errors="coerce"))

# Save to disk
results.to_csv(OUTPUT_CSV, index=False)
print(f"\nProcessing complete. Headway profile saved to {OUTPUT_CSV}")
print("\nTop 10 High-Frequency Corridors:")
print(results[["route_short_name", "route_long_name", "AM Peak (06:00-09:00)", "PM Peak (15:00-19:00)", "is_frequent_network"]].head(10))

Loading GTFS text files...
Active weekday service IDs: [5]
Loaded 113 bus routes and 7615 weekday bus trips.
Parsing arrival times...

Processing complete. Headway profile saved to weekday_bus_headways.csv

Top 10 High-Frequency Corridors:
  route_short_name                      route_long_name  \
0                1   Marietta Blvd/Joseph E Lowery Blvd   
1                2   Ponce de Leon Avenue / Druid Hills   
2                3  Martin Luther King Jr Dr/Auburn Ave   
3                4                      Moreland Avenue   
4                5        Piedmont Road / Sandy Springs   
5                6                 Clifton Road / Emory   
6                8               North Druid Hills Road   
7                9              Boulevard / Tilson Road   
8               12        Howell Mill Road / Cumberland   
9               14              14th Street / Blandtown   

   AM Peak (06:00-09:00)  PM Peak (15:00-19:00)  is_frequent_network  
0                   40.0               

In [3]:
import os
import pandas as pd

# 1. Configuration
GTFS_DIR = r"D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"
OUTPUT_CSV = "scheduled_route_speeds.csv"

# 2. Load the required static tables
routes = pd.read_csv(os.path.join(GTFS_DIR, "routes.txt"))
trips = pd.read_csv(os.path.join(GTFS_DIR, "trips.txt"))
stop_times = pd.read_csv(os.path.join(GTFS_DIR, "stop_times.txt"))

# 3. Isolate standard weekday bus trips
bus_routes = routes[routes["route_type"] == 3].copy()
bus_trips = trips[(trips["route_id"].isin(bus_routes["route_id"])) & (trips["service_id"] == 5)].copy()

# Merge trip IDs with the stop schedules
st = stop_times.merge(bus_trips[["trip_id", "route_id", "direction_id"]], on="trip_id", how="inner")

# 4. Clean GTFS time strings (>24h support)
def parse_gtfs_time(time_str):
    try:
        h, m, s = map(int, str(time_str).strip().split(":"))
        if h >= 24:
            return pd.to_timedelta(f"{h-24:02d}:{m:02d}:{s:02d}") + pd.Timedelta(days=1)
        return pd.to_timedelta(f"{h:02d}:{m:02d}:{s:02d}")
    except Exception:
        return pd.NaT

st["arrival_td"] = st["arrival_time"].apply(parse_gtfs_time)
st["departure_td"] = st["departure_time"].apply(parse_gtfs_time)

# 5. Extract bounds and calculate speed
# Ensure missing origin distances are treated as 0
st["shape_dist_traveled"] = st["shape_dist_traveled"].fillna(0)
st = st.sort_values(["trip_id", "stop_sequence"])

# Identify the start and end of every distinct trip
first_stops = st.groupby("trip_id").first().reset_index()
last_stops = st.groupby("trip_id").last().reset_index()

# Merge origin/destination times and compute elapsed hours and distance
trip_stats = pd.merge(
    first_stops[["trip_id", "route_id", "departure_td"]], 
    last_stops[["trip_id", "arrival_td", "shape_dist_traveled"]], 
    on="trip_id"
)

trip_stats["total_hours"] = (trip_stats["arrival_td"] - trip_stats["departure_td"]).dt.total_seconds() / 3600
trip_stats["speed_mph"] = trip_stats["shape_dist_traveled"] / trip_stats["total_hours"]

# Clean anomalies (0 distance or 0 time)
trip_stats = trip_stats[(trip_stats["total_hours"] > 0) & (trip_stats["shape_dist_traveled"] > 0)]

# 6. Aggregate to the route level
route_speeds = trip_stats.groupby("route_id").agg(
    median_speed_mph=("speed_mph", "median"),
    avg_distance_mi=("shape_dist_traveled", "mean"),
    trip_count=("trip_id", "count")
).reset_index()

# Attach route naming conventions
results = pd.merge(bus_routes[["route_id", "route_short_name", "route_long_name"]], route_speeds, on="route_id", how="inner")
results = results.sort_values(by="median_speed_mph")

# Export
results.to_csv(OUTPUT_CSV, index=False)
print(f"Exported {len(results)} route speed profiles to {OUTPUT_CSV}")

Exported 113 route speed profiles to scheduled_route_speeds.csv


In [4]:
import os
import pandas as pd
import folium

# 1. Configuration & Data Loading
GTFS_DIR = r"D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"

stops = pd.read_csv(os.path.join(GTFS_DIR, "stops.txt"))
stop_times = pd.read_csv(os.path.join(GTFS_DIR, "stop_times.txt"))
trips = pd.read_csv(os.path.join(GTFS_DIR, "trips.txt"))
routes = pd.read_csv(os.path.join(GTFS_DIR, "routes.txt"))

# 2. Isolate a representative trip for Route 110 (Peachtree Road)
route_110 = routes[routes['route_short_name'] == '110'].iloc[0]
trips_110 = trips[
    (trips['route_id'] == route_110['route_id']) & 
    (trips['service_id'] == 5) &  # Standard Weekday
    (trips['direction_id'] == 0)  # Northbound
]
rep_trip_id = trips_110['trip_id'].iloc[0]

# 3. Get stop sequence and spatial coordinates
st_110 = stop_times[stop_times['trip_id'] == rep_trip_id].sort_values('stop_sequence')
st_110 = st_110.merge(stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='left')

# Convert the GTFS shape_dist_traveled (miles) to meters for standard transit evaluation
st_110['shape_dist_traveled'] = st_110['shape_dist_traveled'].fillna(0)
st_110['dist_meters'] = st_110['shape_dist_traveled'] * 1609.34

# Calculate the distance to the NEXT stop
st_110['dist_to_next'] = st_110['dist_meters'].diff().shift(-1)
st_110_valid = st_110.dropna(subset=['dist_to_next']).copy()

# 4. Generate the Interactive Web Map
map_center = [st_110_valid['stop_lat'].mean(), st_110_valid['stop_lon'].mean()]
m = folium.Map(location=map_center, zoom_start=13, tiles='CartoDB positron')

# Draw the route path connecting the stops
points = st_110_valid[['stop_lat', 'stop_lon']].values.tolist()
folium.PolyLine(points, color='#3388ff', weight=4, opacity=0.8).add_to(m)

# Plot and color-code each stop
for idx, row in st_110_valid.iterrows():
    spacing = row['dist_to_next']
    
    # Classify against transit guidelines
    if spacing < 250:
        color = 'red'          # Redundant / Friction Point
    elif spacing > 500:
        color = 'green'        # Stretched spacing
    else:
        color = 'darkorange'   # Optimal
        
    popup_text = (
        f"<div style='font-family: Arial; min-width: 200px;'>"
        f"<b>{row['stop_name']}</b><hr style='margin:4px 0;'/>"
        f"<b>Seq:</b> {row['stop_sequence']}<br>"
        f"<b>Distance to Next Stop:</b> {spacing:.0f} meters<br>"
        f"<b>Status:</b> {'Redundant' if color=='red' else 'Optimal' if color=='darkorange' else 'Stretched'}"
        f"</div>"
    )
    
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Save the deliverable
output_file = "Route110_Stop_Spacing_Map.html"
m.save(output_file)
print(f"Interactive spatial map successfully saved to {output_file}")

# Terminal Output for Analytics
print(f"\n--- Route 110 (Peachtree Road) Analytics ---")
print(f"Total Stops: {len(st_110)}")
print(f"Average Stop Spacing: {st_110_valid['dist_to_next'].mean():.0f} meters")
print(f"Friction Points (< 250m apart): {len(st_110_valid[st_110_valid['dist_to_next'] < 250])} stops")

Interactive spatial map successfully saved to Route110_Stop_Spacing_Map.html

--- Route 110 (Peachtree Road) Analytics ---
Total Stops: 49
Average Stop Spacing: 396 meters
Friction Points (< 250m apart): 9 stops


C:\Users\RASMITHA\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [2]:
import os
import pandas as pd
import folium

# 1. Configuration using your local directory
GTFS_DIR = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"

stops = pd.read_csv(os.path.join(GTFS_DIR, "stops.txt"))
stop_times = pd.read_csv(os.path.join(GTFS_DIR, "stop_times.txt"))
trips = pd.read_csv(os.path.join(GTFS_DIR, "trips.txt"))
routes = pd.read_csv(os.path.join(GTFS_DIR, "routes.txt"))

# 2. Isolate a representative trip for Route 110 (Peachtree Road)
route_110 = routes[routes['route_short_name'] == '110'].iloc[0]
trips_110 = trips[
    (trips['route_id'] == route_110['route_id']) & 
    (trips['service_id'] == 5) &  # Standard Weekday
    (trips['direction_id'] == 0)  # Northbound
]
rep_trip_id = trips_110['trip_id'].iloc[0]

# 3. Get stop sequence and spatial coordinates
st_110 = stop_times[stop_times['trip_id'] == rep_trip_id].sort_values('stop_sequence')
st_110 = st_110.merge(stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='left')

# Convert shape_dist_traveled (miles) to meters for standard transit evaluation
st_110['shape_dist_traveled'] = st_110['shape_dist_traveled'].fillna(0)
st_110['dist_meters'] = st_110['shape_dist_traveled'] * 1609.34

# Calculate the distance to the NEXT stop
st_110['dist_to_next'] = st_110['dist_meters'].diff().shift(-1)
st_110_valid = st_110.dropna(subset=['dist_to_next']).copy()

# 4. Generate the Interactive Map (Free CartoDB Background)
map_center = [st_110_valid['stop_lat'].mean(), st_110_valid['stop_lon'].mean()]

# Use CartoDB Positron for a free, high-contrast background without an API key
m = folium.Map(location=map_center, zoom_start=13, tiles='CartoDB positron')

# Draw the route path
points = st_110_valid[['stop_lat', 'stop_lon']].values.tolist()
folium.PolyLine(points, color='#3388ff', weight=4, opacity=0.8).add_to(m)

# Plot and color-code each stop based on spacing guidelines
for idx, row in st_110_valid.iterrows():
    spacing = row['dist_to_next']
    
    if spacing < 250:
        color = 'red'          # Redundant / Friction Point
    elif spacing > 500:
        color = 'green'        # Stretched spacing
    else:
        color = 'darkorange'   # Optimal
        
    popup_text = (
        f"<div style='font-family: Arial; min-width: 200px;'>"
        f"<b>{row['stop_name']}</b><hr style='margin:4px 0;'/>"
        f"<b>Seq:</b> {row['stop_sequence']}<br>"
        f"<b>Distance to Next Stop:</b> {spacing:.0f} meters<br>"
        f"</div>"
    )
    
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Print Analytics to Terminal
print(f"--- Route 110 (Peachtree Road) Analytics ---")
print(f"Total Stops: {len(st_110)}")
print(f"Average Stop Spacing: {st_110_valid['dist_to_next'].mean():.0f} meters")
print(f"Friction Points (< 250m apart): {len(st_110_valid[st_110_valid['dist_to_next'] < 250])} stops")

# Display the map directly in VS Code (Jupyter / Interactive Window)
m = folium.Map(location=map_center, zoom_start=13, tiles='OpenStreetMap')

--- Route 110 (Peachtree Road) Analytics ---
Total Stops: 49
Average Stop Spacing: 396 meters
Friction Points (< 250m apart): 9 stops


C:\Users\RASMITHA\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [4]:
pip install --upgrade plotly

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\RASMITHA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import os
import pandas as pd
import plotly.express as px

# 1. Configuration & Data Loading
GTFS_DIR = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"

stops = pd.read_csv(os.path.join(GTFS_DIR, "stops.txt"))
stop_times = pd.read_csv(os.path.join(GTFS_DIR, "stop_times.txt"))
trips = pd.read_csv(os.path.join(GTFS_DIR, "trips.txt"))
routes = pd.read_csv(os.path.join(GTFS_DIR, "routes.txt"))

# 2. Isolate a representative trip for Route 110 (Peachtree Road)
route_110 = routes[routes['route_short_name'] == '110'].iloc[0]
trips_110 = trips[
    (trips['route_id'] == route_110['route_id']) & 
    (trips['service_id'] == 5) &  # Standard Weekday
    (trips['direction_id'] == 0)  # Northbound
]
rep_trip_id = trips_110['trip_id'].iloc[0]

# 3. Get stop sequence and spatial coordinates
st_110 = stop_times[stop_times['trip_id'] == rep_trip_id].sort_values('stop_sequence')
st_110 = st_110.merge(stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='left')

# Convert shape_dist_traveled (miles) to meters
st_110['shape_dist_traveled'] = st_110['shape_dist_traveled'].fillna(0)
st_110['dist_meters'] = st_110['shape_dist_traveled'] * 1609.34

# Calculate the distance to the NEXT stop
st_110['dist_to_next'] = st_110['dist_meters'].diff().shift(-1)
st_110_valid = st_110.dropna(subset=['dist_to_next']).copy()

# 4. Classify Spacing for Plotly Colors
def classify_spacing(dist):
    if dist < 250: return 'Redundant (<250m)'
    elif dist > 500: return 'Stretched (>500m)'
    else: return 'Optimal (250-500m)'

st_110_valid['Spacing Status'] = st_110_valid['dist_to_next'].apply(classify_spacing)

# 5. Generate the Interactive Plotly Map (Renders directly in VS Code)
import plotly.graph_objects as go

# Map colors to the status
color_map = {
    'Redundant (<250m)': 'red',
    'Optimal (250-500m)': 'orange',
    'Stretched (>500m)': 'green'
}
st_110_valid['color'] = st_110_valid['Spacing Status'].map(color_map)

# 5. Generate the Interactive Plotly Map (Graph Objects method)
fig = go.Figure(go.Scattermapbox(
    lat=st_110_valid['stop_lat'],
    lon=st_110_valid['stop_lon'],
    mode='markers',
    marker=go.scattermapbox.Marker(
        size=9,
        color=st_110_valid['color'],
        opacity=0.9
    ),
    text=st_110_valid['stop_name'] + "<br>Next Stop: " + st_110_valid['dist_to_next'].astype(int).astype(str) + "m",
    hoverinfo='text'
))

fig.update_layout(
    title="Route 110: Bus Stop Spacing Optimization",
    mapbox_style="open-street-map",
    mapbox=dict(
        center=dict(lat=st_110_valid['stop_lat'].mean(), lon=st_110_valid['stop_lon'].mean()),
        zoom=11.5
    ),
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig.show()

# Print Analytics to Terminal
print(f"--- Route 110 Analytics ---")
print(f"Total Stops: {len(st_110)}")
print(f"Average Stop Spacing: {st_110_valid['dist_to_next'].mean():.0f} meters")
print(f"Friction Points (< 250m apart): {len(st_110_valid[st_110_valid['dist_to_next'] < 250])} stops")

AttributeError: module 'plotly.graph_objects' has no attribute 'Scattermapbox'